In [30]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
env = gym.make('CartPole-v1', render_mode="rgb_array")

epsilon = 1
episodes = 1000
discount = 0.95
step_size = 0.01
batch_size = 64
losses = []

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = Net()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=step_size)
buffer = []

def show_ep():
    frame = env.render()
    plt.imshow(frame)
    plt.axis("off")
    plt.show()

for ep in range(episodes):
    done = False
    obs, info = env.reset()
    t = 0
    # obs, q_value  t-100 -> t
    avg_loss = 0
    epsilon = max(0.05, 1-ep/episodes)
    if ep>0:
        print(ep, losses[-1])

    while not done:
        #show_ep()
        t += 1
        pred = model(torch.tensor(obs).unsqueeze(0))

        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = torch.argmax(pred).item()

        save_obs = obs
        obs, reward, terminated, truncated, info = env.step(action)
        done = truncated or terminated

        buffer.append((save_obs, reward, action, obs, done))
        buffer = buffer[-10000:]

        if len(buffer)>=batch_size:
            sample = np.random.choice(range(len(buffer)), size=min(batch_size, len(buffer)), replace=False)
            batch = [buffer[i] for i in sample]

            states, rewards, actions, next_states, dones = zip(*batch)
            states = torch.tensor(np.array(states), dtype=torch.float32)
            rewards = torch.tensor(rewards, dtype=torch.float32)
            actions = torch.tensor(actions, dtype=torch.int64)
            next_states = torch.tensor(np.array(next_states), dtype=torch.float32)
            dones = torch.tensor(dones, dtype=torch.float32)

            targets = [rewards[i] if dones[i] else rewards[i] +discount * torch.max(model(next_states[i])) for i in range(len(rewards))]
            targets = torch.tensor(targets)
            preds = model(states).gather(1, actions.unsqueeze(1))
            targets = targets.unsqueeze(1)

            loss = criterion(preds, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            avg_loss = (avg_loss*(t-1)+np.mean(loss.item()))/t

        losses.append(avg_loss)

    if ep%100==0:
        done = False
        obs, info = env.reset()
        while not done:
            pred = model(torch.tensor(obs).unsqueeze(0))
            obs, reward, terminated, truncated, info = env.step(torch.argmax(pred).item())
            done = truncated or terminated
            show_ep()

window = 50
x = range(0,episodes,window)
avg_train = [float(np.mean(losses[i:i+window])) for i in x]
plt.plot(x,avg_train)
plt.show()

done = False
obs, info = env.reset()
while not done:
    pred = model(torch.tensor(obs).unsqueeze(0))
    obs, reward, terminated, truncated, info = env.step(torch.argmax(pred).item())
    done = truncated or terminated
    show_ep()

Output hidden; open in https://colab.research.google.com to view.